# 04 · From automatic labels to a fit probability

**Goal:** train a model on the automatic labels from Step 2, so that every candidate gets a **fit between 0 and 1**, and rank by it.

**Why a model, when we already have a similarity score?**
1. **The brief asks for fit as a probability.** A similarity score isn't one.
2. **A model can learn from feedback.** In Step 5, the expert's stars become new training examples. A fixed similarity score can't learn.
3. **A model learns a weight for every word.** It can pick up words that are common among fits but aren't in the keywords.

**Questions:** which model, which input (TF-IDF words, GloVe meaning, or both), and how good is it, measured honestly with only 34 labelled profiles?

In [1]:
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.metrics import log_loss, roc_auc_score

from talent_ranker.config import KEYWORDS
from talent_ranker.data import deduplicate, load_candidates
from talent_ranker.embeddings import load_word_vectors
from talent_ranker.evaluation import bootstrap_stability, out_of_fold_fit
from talent_ranker.labelling import auto_label
from talent_ranker.model import (
    combine_features,
    glove_features,
    predict_fit,
    tfidf_features,
    train_fit_model,
)
from talent_ranker.similarity import tfidf_similarity

pd.set_option("display.max_colwidth", 80)
profiles = deduplicate(load_candidates())
profiles["similarity"] = tfidf_similarity(profiles["job_title"], KEYWORDS)
profiles["auto_label"] = auto_label(profiles["similarity"])
titles = profiles["job_title"]
labels = profiles["auto_label"]
print(
    f"{len(profiles)} profiles: {labels.eq(1).sum()} fit, {labels.eq(0).sum()} not a fit, "
    f"{labels.isna().sum()} unsure"
)

53 profiles: 16 fit, 18 not a fit, 19 unsure


## 1. The model: logistic regression

| Model | Verdict | Reason |
|---|---|---|
| **Logistic regression** | ✅ chosen | Outputs a probability (0–1) directly; regularization keeps it sensible with 34 examples; trains in milliseconds, so it can retrain after every star; one interpretable weight per word |
| Naive Bayes | ❌ | Assumes words are independent, which pushes probabilities to extremes, and can't take GloVe's negative numbers |
| SVM | ❌ | Gives a score, not a probability, and calibrating one needs extra labelled data |
| Random forest, boosting | ❌ | Memorise 34 examples, and give step-like rather than smooth probabilities |
| k-nearest neighbours | ❌ | No weights to inspect, depends on k, and has few neighbours |
| Neural network, learning-to-rank | ❌ | Need far more data, or many searches with graded judgements |

## 2. Three possible inputs
- **TF-IDF words:** one column per word, the same settings as the Step 2 similarity.
- **GloVe meaning:** the average word vector from Step 3 (100 numbers).
- **Both:** side by side, scaled so every row still has length 1, which makes the comparison fair (see below).

In [2]:
vectors = load_word_vectors()
word_features, words = tfidf_features(titles)
meaning_features = glove_features(titles, vectors)
inputs = {
    "TF-IDF words": word_features,
    "GloVe meaning": meaning_features,
    "both": combine_features(word_features, meaning_features),
}
{name: features.shape for name, features in inputs.items()}

{'TF-IDF words': (53, 180), 'GloVe meaning': (53, 100), 'both': (53, 280)}

53 profiles: **180** word columns, **100** meaning columns, **280** combined.

## 3. Honest evaluation with only 34 labels

| Choice | Reason |
|---|---|
| **Out-of-fold predictions** | Each labelled profile is scored by a model that never saw it. With 34 examples, we can't spare a separate test set |
| **Grouped by title** | Two different people share the exact title "Aspiring Human Resources Professional". Split across training and testing, they would make the test trivially easy (leakage) |
| **Stratified, 5 folds, 5 different shuffles** | Every fold has both classes, and repeating the shuffle shows whether a result depends on one lucky split |
| **Log loss** (lower is better) | Judges probabilities: rewards being right *and* confident, and punishes confident mistakes. It still separates inputs when AUC is already perfect |
| **AUC** | A sanity check that the model separates the two groups |
| ~~Accuracy~~ | Would be 100% for everything on perfectly separable labels, so it's meaningless here |

⚠️ These checks measure agreement with the *automatic* labels, which TF-IDF made, so they lean towards TF-IDF. The independent check is the expert, in Step 5.

In [3]:
labelled = labels.notna()
y = labels[labelled].astype(int)
candidates = {
    **inputs,
    "both, not rescaled": sparse.hstack(list(inputs.values())[:2]).tocsr(),
}
results = {}
for name, features in candidates.items():
    runs = [
        out_of_fold_fit(features, labels, groups=titles, random_state=seed)
        for seed in range(5)
    ]
    losses = [log_loss(y, oof) for oof in runs]
    results[name] = {
        "AUC (mean)": np.mean([roc_auc_score(y, oof) for oof in runs]),
        "log loss (mean)": np.mean(losses),
        "log loss (best split)": min(losses),
        "log loss (worst split)": max(losses),
    }
pd.DataFrame(results).T.round(3)

,AUC (mean),log loss (mean),log loss (best split),log loss (worst split)
TF-IDF words,1.000,0.477,0.470,0.484
GloVe meaning,0.997,0.505,0.502,0.512
both,1.000,0.487,0.482,0.494
"both, not rescaled",1.000,0.403,0.397,0.411


**Finding:**

| Input | AUC | Log loss (mean, range over 5 splits) |
|---|---|---|
| **TF-IDF words** | 1.000 | **0.477** (0.470–0.484) |
| both (fair, rescaled) | 1.000 | 0.487 (0.482–0.494) |
| GloVe meaning | 0.997 | 0.505 (0.502–0.512) |
| both, *not* rescaled | 1.000 | 0.403 |

- **Every input separates the labels almost perfectly (AUC ≈ 1)**, so AUC can't choose. Log loss can.
- **TF-IDF words are best, and in every split.** The worst TF-IDF split (0.484) beats the best "both" split (0.482)… almost: the ranges barely touch. The order is the same in all 5 shuffles.
- **The trap:** "both, not rescaled" seems clearly best (0.403), but only because its rows are longer (length √2), and longer rows alone make logistic regression more confident. Rescaled fairly, the advantage disappears. **Always compare inputs on equal terms.**

## 4. Why the regularization strength C isn't tuned
C controls how strongly the model is held back. A larger C means bolder, more confident probabilities.

In [4]:
by_strength = {}
for C in [0.1, 1.0, 10.0, 100.0]:
    oof = out_of_fold_fit(word_features, labels, groups=titles, C=C)
    by_strength[f"C = {C}"] = log_loss(y, oof)
pd.Series(by_strength, name="log loss, TF-IDF words").round(3)

C = 0.1      0.655
C = 1.0      0.471
C = 10.0     0.231
C = 100.0    0.103
Name: log loss, TF-IDF words, dtype: float64

**Finding: log loss keeps falling as C grows (0.655 → 0.471 → 0.231 → 0.103), with no minimum.**

The automatic labels can be separated perfectly, so the bolder the model, the better it "agrees" with them. Tuning C would pick a model that copies the labels with near-certainty, **including their known mistakes**: the C.T. Bauer graduate, for example, would be confidently scored low.

**Decision: keep the default C = 1.0.** Its probabilities stay moderate, which leaves room for the expert's stars to move them in Step 5.

## 5. The hard cases from Step 1, for each input
Rank 1 is best; ties share a rank.

In [5]:
hard_cases = {
    "2019 C.T. Bauer": "aspiring, long title",
    "Retired Army": "seeking, long title",
    "Human Resources Management Major": "HR student",
    "Student at Humber College": "aspiring student",
    "Experienced Retail Manager": "aspiring career changer",
    "Generalist at Loparex": "working in HR",
    "HR Senior Specialist": "working in HR",
    "Director Human Resources at EY": "working in HR, senior",
    "Nortia": "job ad",
    "Native English Teacher": "not HR",
}
fits = pd.DataFrame(
    {
        name: predict_fit(train_fit_model(features, labels), features, profiles.index)
        for name, features in inputs.items()
    }
)
ranks = fits.rank(ascending=False, method="min").astype(int)
rows = []
for text, kind in hard_cases.items():
    i = profiles.index[titles.str.contains(text, regex=False)][0]
    rows.append({"title": titles[i][:45], "what it is": kind, **ranks.loc[i].to_dict()})
pd.DataFrame(rows)

,title,what it is,TF-IDF words,GloVe meaning,both
0,2019 C.T. Bauer College of Business Graduate,"aspiring, long title",23,42,35
1,"Retired Army National Guard Recruiter, office","seeking, long title",27,33,30
2,Human Resources Management Major,HR student,16,17,17
3,Student at Humber College and Aspiring Human,aspiring student,17,30,22
4,Experienced Retail Manager and aspiring Human,aspiring career changer,9,19,9
5,Human Resources Generalist at Loparex,working in HR,19,1,10
6,HR Senior Specialist,working in HR,22,22,24
7,Director Human Resources at EY,"working in HR, senior",28,24,27
8,"Nortia Staffing is seeking Human Resources, P",job ad,30,18,26
9,Native English Teacher at EPIK (English Progr,not HR,38,43,43


**Finding: GloVe teaches the model "HR or not", not "aspiring or not".**

| Case | TF-IDF words | GloVe meaning | both |
|---|---|---|---|
| HR Generalist (working in HR) | 19 | **1** | 10 |
| Aspiring career changer | 9 | 19 | 9 |
| Aspiring student (Humber) | 17 | 30 | 22 |
| C.T. Bauer graduate (aspiring) | 23 | 42 | 35 |
| Nortia job ad | 30 | 18 | 26 |

- With GloVe, a **working HR Generalist becomes the #1 candidate** for an *aspiring* search, and aspiring people drop by 10–20 places.
- Every confident "not a fit" example is a non-HR title, so meaning features learn the HR-vs-non-HR boundary, and that is what they rank by.
- TF-IDF words keep the career-stage words ("aspiring", "seeking") in charge.

## 6. Robustness: does the ranking survive small changes in the training labels?
We retrain on 200 resampled versions of the 34 labels (drawn with replacement) and compare each new ranking with the original one (Spearman: 1 = identical order).

In [6]:
stability = {
    name: bootstrap_stability(features, labels) for name, features in inputs.items()
}
pd.DataFrame(
    {
        name: {"median rank agreement": np.median(c), "worst 5%": np.percentile(c, 5)}
        for name, c in stability.items()
    }
).T.round(3)

,median rank agreement,worst 5%
TF-IDF words,0.955,0.932
GloVe meaning,0.988,0.967
both,0.970,0.944


**Finding: all three are robust.** Median agreement is 0.955 for TF-IDF, 0.970 for both and 0.988 for GloVe, and even the worst 5% of resamples stay above 0.93.

GloVe is the *most* stable, but stability isn't correctness: it is stably ranking the wrong thing for this search (working HR above aspiring).

### Decision
**Logistic regression on TF-IDF words, with C = 1.0.** It has the best held-out log loss in every split, it keeps the career stage in charge, and it's robust (0.955).

Word vectors were checked, and **not applied here**. For this search the career-stage words matter more than general meaning. They remain an option for roles where synonyms matter more than exact words.

## 7. The final model's ranking

In [7]:
model = train_fit_model(word_features, labels)
profiles["fit"] = predict_fit(model, word_features, profiles.index)
ranked = profiles.sort_values("fit", ascending=False).assign(
    rank=range(1, len(profiles) + 1)
)
ranked[["rank", "job_title", "similarity", "auto_label", "fit"]].head(15).round(2)

,rank,job_title,similarity,auto_label,fit
2,1,Aspiring Human Resources Professional,0.76,1,0.80
45,2,Aspiring Human Resources Professional,0.76,1,0.80
21,3,"Aspiring Human Resources Manager, seeking internship in Human Resources.",0.63,1,0.79
22,4,Human Resources Professional,0.39,1,0.73
5,5,Aspiring Human Resources Specialist,0.68,1,0.70
13,6,Seeking Human Resources Opportunities,0.67,1,0.70
47,7,Seeking Human Resources Position,0.67,1,0.69
48,8,Aspiring Human Resources Manager Graduating May 2020 Seeking an Entry-Level ...,0.36,1,0.67
14,9,Experienced Retail Manager and aspiring Human Resources Professional,0.40,1,0.67
12,10,Aspiring Human Resources Management student seeking an internship,0.46,1,0.67


**Finding:** the top 15 are all "fit" by the automatic labels, led by "Aspiring Human Resources Professional" (fit **0.80**).

One visible change from plain similarity: **"Human Resources Professional"**, someone already working in HR, rises from rank 11 by similarity to **rank 4** by fit. The next cell shows why.

## 8. What did the model learn?
Each word has one weight: positive pushes fit up, negative pushes it down.

In [8]:
weights = pd.Series(model.coef_[0], index=words).sort_values()
pd.DataFrame(
    {
        "pushes fit UP": weights.index[::-1][:8],
        "weight": weights.to_numpy()[::-1][:8].round(2),
        "pushes fit DOWN": weights.index[:8],
        "weight ": weights.to_numpy()[:8].round(2),
    }
)

,pushes fit UP,weight,pushes fit DOWN,weight
0,resources,1.21,university,-0.54
1,human,1.21,student,-0.29
2,aspiring,1.15,information,-0.22
3,seeking,0.83,systems,-0.22
4,professional,0.70,set,-0.20
5,opportunities,0.38,success,-0.20
6,manager,0.35,english,-0.19
7,major,0.33,chapman,-0.17


**Finding: the model learned the keywords, plus a few artefacts of automatic labelling.**
- **Up:** resources (1.21), human (1.21), aspiring (1.15) and seeking (0.83), which are the keywords, as expected.
- **"professional" (+0.70)** was learned because many fit titles say "Aspiring Human Resources *Professional*". That's why "Human Resources Professional" ranks 4th, although that person is already working.
- **"student" (−0.29) and "university" (−0.54) push fit *down*.** Bare titles like "Student" and "Student at Chapman University" share no keyword word, so they were labelled "not a fit". For an *aspiring* search, being a student should, if anything, be a plus.

These are exactly the mistakes a domain expert would correct, which is the purpose of Step 5.

In [9]:
agreement = profiles["fit"].corr(profiles["similarity"], method="spearman")
print(f"Rank agreement between fit and the TF-IDF similarity: {agreement:.3f}")
print(f"fit ranges from {profiles['fit'].min():.2f} to {profiles['fit'].max():.2f}")

Rank agreement between fit and the TF-IDF similarity: 0.951
fit ranges from 0.24 to 0.80


**Finding: a model can't know more than its labels.** Its ranking agrees with the TF-IDF similarity at **0.951**. What it adds:
- **a probability** (0.24–0.80);
- **word weights** we can inspect;
- **the ability to learn from feedback.**

Its view of "fit" is still TF-IDF's view, artefacts included.

**What "fit = 0.80" means here:** "very likely to be labelled a fit by the automatic rule". It is **not** the probability of being hired. Calibrating that would need real outcomes (placements).

## 9. Does the 0.3 threshold from Step 2 drive the results?
We retrain with different thresholds and compare each ranking with the 0.30 one.

In [10]:
sensitivity = {}
for threshold in [0.25, 0.30, 0.35, 0.40]:
    alt_labels = auto_label(profiles["similarity"], fit_min=threshold)
    alt_fit = predict_fit(
        train_fit_model(word_features, alt_labels), word_features, profiles.index
    )
    sensitivity[threshold] = {
        "labelled fit": int(alt_labels.eq(1).sum()),
        "unsure": int(alt_labels.isna().sum()),
        "rank agreement with 0.30": alt_fit.corr(profiles["fit"], method="spearman"),
    }
pd.DataFrame(sensitivity).T.round(3)

,labelled fit,unsure,rank agreement with 0.30
0.25,21.0,14.0,0.985
0.30,16.0,19.0,1.000
0.35,15.0,20.0,0.994
0.40,10.0,25.0,0.989


**Finding: no.** Moving the threshold from 0.25 to 0.40 halves the number of "fit" labels (21 → 10), but the ranking barely moves: agreement is **0.984–0.994**. The threshold choice from Step 2 isn't what drives the results.

## 10. Output: a fit for each of the 104 original rows
Each profile's fit and rank go back to every original id, so duplicates share their profile's score.

In [11]:
per_row = (
    ranked.explode("ids")
    .rename(columns={"ids": "id"})[["id", "job_title", "fit", "rank"]]
    .reset_index(drop=True)
)
print(f"{len(per_row)} rows, one per original id")
per_row.head(10).round(3)

104 rows, one per original id


,id,job_title,fit,rank
0,3,Aspiring Human Resources Professional,0.796,1
1,17,Aspiring Human Resources Professional,0.796,1
2,21,Aspiring Human Resources Professional,0.796,1
3,33,Aspiring Human Resources Professional,0.796,1
4,46,Aspiring Human Resources Professional,0.796,1
5,58,Aspiring Human Resources Professional,0.796,1
6,97,Aspiring Human Resources Professional,0.796,2
7,73,"Aspiring Human Resources Manager, seeking internship in Human Resources.",0.790,3
8,74,Human Resources Professional,0.728,4
9,6,Aspiring Human Resources Specialist,0.698,5


**All 104 original ids** get a fit and a rank. For example, ids 3, 17, 21, 33, 46 and 58 are the same person and share rank 1 (fit 0.796).

## Summary

| Question | Answer |
|---|---|
| Model | **Logistic regression**: probability output, works with few examples, retrains instantly, interpretable |
| Input | **TF-IDF words**: best held-out log loss (0.477 vs 0.487 and 0.505) in every split, and it keeps career stage in charge. GloVe ranks a working HR Generalist #1 |
| Lesson | Compare inputs on equal terms: the unscaled combination looked best (0.403) for a reason unrelated to quality |
| Regularization | C = 1.0, deliberately not tuned (tuning on separable automatic labels gives overconfidence) |
| Robustness | Stable under resampled labels (0.955) and under the labelling threshold (≥ 0.984) |
| Limits | The model mirrors its labels (0.951 agreement with similarity) and learned artefacts: "student" ↓, "professional" ↑ |
| Output | fit (0.24–0.80) and rank for all 104 ids |

**Next (Step 5):** the domain-expert step. A simulated expert checks the top of the list and **stars** an ideal candidate. The model retrains with that star, and with the candidates the expert passed over, and re-ranks. We measure how the ranking improves with each star, and re-check the TF-IDF vs GloVe choice with feedback that doesn't come from TF-IDF.